In [ ]:
import ast
import os
import pickle
import numpy as np
import pandas as pd
from wildlife_tools.similarity.wildfusion import WildFusion
from animal_re_id.config import load_paths, resolve_path
from animal_re_id.datasets import TurtlesOfSMSRC, get_wildlife_dataset
from animal_re_id.models import best_matches_discovery, best_matches_matching, gather_df, top_indices
from animal_re_id.models import get_pipeline_miew, get_pipeline_aliked, get_pipeline_disk, get_pipeline_superpoint
from animal_re_id.similarity import MetadataIgnorePairsPolicy
import warnings
warnings.filterwarnings("ignore")

In [ ]:
def file_name_pkl(root_results, method_name, B, img_load, matching_part):
    file_name_part = f"{method_name}_{B}_{img_load}_{matching_part}"
    return f"{root_results}/{file_name_part}.pkl"

def load_data(file_name):
    with open(file_name, "rb") as file:
        saved_vars = pickle.load(file)
    similarity = saved_vars["similarity"]
    query = saved_vars["query"]
    database = saved_vars["database"]
    return similarity, query, database

In [ ]:
local_path = "../../config/paths.json"
paths = load_paths("TurtlesOfSMSRC", local_path=local_path)

B_max = 100
max_per_method = 80
max_per_identity = 2

root = resolve_path(paths["root"])
dataset_name = 'TurtlesOfSMSRC'
matching_parts = ['heads', 'front flippers', 'hind flippers']
path_replace_identities = "hidden/matched_encounters.txt"
img_load = "bbox"
device = "cuda"
get_priority = get_pipeline_miew
get_pipelines = [get_pipeline_aliked, get_pipeline_disk, get_pipeline_superpoint]
method_names = ['miew-aliked', 'miew-disk', 'miew-superpoint']

In [ ]:
root_features = resolve_path(paths["root_features"])
root_results = resolve_path(paths["root_similarity"])

In [ ]:
with open(path_replace_identities) as f:
    replace_identity = ast.literal_eval(f.read())
    
os.makedirs(root_results, exist_ok=True)
dataset_kwargs = {"img_load": img_load, "replace_identity": replace_identity}
database_query_matching = False
dataset = TurtlesOfSMSRC(root, database_query_matching, dataset_kwargs=dataset_kwargs)

In [ ]:
B = 100
similarities = {}
for matching_part in matching_parts:
    # Get the file names for similarity matrices
    file_names_save = [file_name_pkl(root_results, method_name, B, img_load, matching_part) for method_name in method_names]

    # If the similarity matrices already exist, skip the computation
    if np.all([os.path.exists(file_name_save) for file_name_save in file_names_save]):
        continue

    # Get the pipelines
    cache_prefix = f"{root_features}/{matching_part}_{img_load}"
    pipeline_priority = get_priority(device=device, cache_prefix=cache_prefix)
    pipelines = [get_pipeline(device=device, cache_prefix=cache_prefix) for get_pipeline in get_pipelines]

    # Extract all necessary computation
    query = get_wildlife_dataset(dataset, matching_part)
    database = query
    pair_selector = MetadataIgnorePairsPolicy(cache_key=None).get_pair_selector(dataset, matching_part)

    for pipeline, file_name_save in zip(pipelines, file_names_save):
        # If the similarity matrix already exists, skip the computation
        if os.path.exists(file_name_save):
            continue

        # Compute similarity for each pipeline
        wildfusion = WildFusion(calibrated_pipelines=[pipeline], priority_pipeline=pipeline_priority, pair_selector=pair_selector)
        similarity = wildfusion(query, database, B=B)
        similarity = similarity.astype(np.float64)

        # Save the data
        save_data = {
            "similarity": similarity,
            "query": query,
            "database": database,
            "B": B,
            "img_load": img_load,
            "matching_part": matching_part
        }
        with open(file_name_save, "wb") as file:
            pickle.dump(save_data, file)

In [ ]:
results_all = {}
query_all = {}
database_all = {}
for matching_part in matching_parts:
    results = None
    for method_name in method_names:
        file_name_save = file_name_pkl(root_results, method_name, B, img_load, matching_part)
        similarity, query, database = load_data(file_name_save)
        
        ignore_lower_diagonal = False if database_query_matching else True
        i_sort, j_sort = top_indices(similarity, ignore_lower_diagonal=ignore_lower_diagonal)
        result = gather_df(query, database, i_sort, j_sort, similarity, method_name, matching_part)
        results = pd.concat((results, result))

    # TODO: unify it somehow so that it is general everywhere
    max_per_encounter = 4
    if database_query_matching:
        results_all[matching_part] = best_matches_matching(results, max_per_encounter)
    else:
        results_all[matching_part] = best_matches_discovery(results, max_per_method*len(method_names), max_per_identity)

In [ ]:
results_save = pd.concat(results_all.values())
if 'key' in results_save.columns:
    results_save = results_save.drop(['key'], axis=1)
results_save.to_csv(f'../../verification_data_{dataset_name}.csv', index=False)

In [ ]:
from animal_re_id.models import plot_matches

dataset = TurtlesOfSMSRC(root, database_query_matching, dataset_kwargs=dataset_kwargs)
query_all = dataset.wildlife_dataset

n_plot = 5
for matching_part in matching_parts:
    for _, result in results_all[matching_part].iloc[:n_plot].iterrows():
        plot_matches(query_all, result)